# 04 · Theater — UMAP parameter animations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/splevine/clustering-good-bad-beautiful/blob/main/notebooks/04_theater.ipynb)

**Eye candy for the talk.** Sweep UMAP's `n_neighbors` and watch clusters form, dissolve, and re-form as the local/global balance shifts. Same movies, same embeddings — only the hyperparameter changes.

We render two animations: one on **overview embeddings** (text), one on **poster embeddings** (CLIP). They're saved as MP4 at the repo root so the landing page can embed them in `<video>` tags.

## Setup

In [ ]:
# Colab: uncomment to install
# !pip install -q umap-learn plotly matplotlib imageio-ffmpeg pandas pyarrow tqdm

import sys
from pathlib import Path

import imageio_ffmpeg
import matplotlib
import numpy as np
import pandas as pd
from IPython.display import HTML

matplotlib.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()

sys.path.insert(0, str(Path.cwd()))  # so `import viz` works when notebook is in notebooks/
import viz

## Load movies, embeddings, and labels

We color points by **primary genre**, bucketing anything outside the top 8 into "Other" so the color palette stays legible.

In [ ]:
EMB_DIR = Path("../embeddings")
text_emb = np.load(EMB_DIR / "overviews_minilm.npy")
clip_emb = np.load(EMB_DIR / "posters_clip_b32.npy")

all_movies = pd.read_parquet("../data/movies.parquet")
text_movies = all_movies[all_movies["overview"].str.len() > 30].reset_index(drop=True)
assert len(text_movies) == len(text_emb), (len(text_movies), len(text_emb))

poster_dir = Path("../data/posters")
poster_movies = all_movies.copy()
poster_movies["poster_file"] = poster_movies["id"].map(lambda i: poster_dir / f"{i}.jpg")
poster_movies = poster_movies[poster_movies["poster_file"].map(lambda p: p.exists())].reset_index(drop=True)
assert len(poster_movies) == len(clip_emb), (len(poster_movies), len(clip_emb))

print(f"text: {text_emb.shape}   poster: {clip_emb.shape}")

In [ ]:
TOP_GENRES = 8

def genre_codes(df):
    primary = df["genres"].map(lambda gs: gs[0] if len(gs) else "Other")
    top = primary.value_counts().head(TOP_GENRES).index.tolist()
    collapsed = primary.where(primary.isin(top), "Other")
    labels = sorted(collapsed.unique())
    code_map = {label: i for i, label in enumerate(labels)}
    codes = collapsed.map(code_map).to_numpy()
    return codes, labels

text_codes, text_labels = genre_codes(text_movies)
poster_codes, poster_labels = genre_codes(poster_movies)
print("text genre legend:  ", text_labels)
print("poster genre legend:", poster_labels)

## Subsample for animation speed

A 5,000-point animation rendered frame-by-frame is slow and produces a big video. 2,000 points is enough to see cluster structure while keeping render time reasonable.

In [ ]:
N_SAMPLE = 2000
rng = np.random.default_rng(0)

def stratified_sample(codes, n):
    # Take equal-ish counts per class so minority genres aren't crushed.
    per_class = max(1, n // len(np.unique(codes)))
    picks = []
    for c in np.unique(codes):
        idx = np.where(codes == c)[0]
        picks.extend(rng.choice(idx, size=min(per_class, len(idx)), replace=False))
    picks = np.array(picks)
    rng.shuffle(picks)
    return picks[:n]

text_idx = stratified_sample(text_codes, N_SAMPLE)
poster_idx = stratified_sample(poster_codes, N_SAMPLE)
print(f"text sample:   {len(text_idx)}")
print(f"poster sample: {len(poster_idx)}")

## 1. Animation on movie overview embeddings

Sweep `n_neighbors` through a wide range. Small values exaggerate local structure; large values pull everything into a global manifold. The morph between is where the audience *gets* what UMAP's key knob does.

In [ ]:
param_values = [3, 5, 10, 15, 30, 60]

anim_text = viz.create_umap_animation(
    data=text_emb[text_idx],
    labels=text_codes[text_idx],
    param_name="n_neighbors",
    param_values=param_values,
    n_components=3,
    n_static_frames=15,
    n_tween_frames=10,
    rotation_speed=2.0,
    min_dist=0.1,
    metric="cosine",
    random_state=0,
)

text_mp4 = Path("../animation_text.mp4")
anim_text.save(str(text_mp4), writer="ffmpeg", fps=20, dpi=100, bitrate=2400)
print(f"wrote {text_mp4} ({text_mp4.stat().st_size // 1024} KB)")

In [ ]:
# Preview inline (this produces a large HTML blob; skip if you just want the MP4)
HTML(anim_text.to_jshtml(fps=20))

## 2. Animation on movie poster embeddings (CLIP)

Same sweep, different modality. The audience can directly compare how text and image spaces behave under the same parameter settings.

In [ ]:
anim_posters = viz.create_umap_animation(
    data=clip_emb[poster_idx],
    labels=poster_codes[poster_idx],
    param_name="n_neighbors",
    param_values=param_values,
    n_components=3,
    n_static_frames=15,
    n_tween_frames=10,
    rotation_speed=2.0,
    min_dist=0.1,
    metric="cosine",
    random_state=0,
)

posters_mp4 = Path("../animation_posters.mp4")
anim_posters.save(str(posters_mp4), writer="ffmpeg", fps=20, dpi=100, bitrate=2400)
print(f"wrote {posters_mp4} ({posters_mp4.stat().st_size // 1024} KB)")

## 3. Bonus — 3D Plotly scatter you can rotate

The animations are the showstopper; this is the "let the audience play" artifact.

In [ ]:
from umap import UMAP

embedding_3d = UMAP(n_components=3, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=0).fit_transform(text_emb[text_idx])
fig = viz.create_3d_scatter(embedding_3d, text_codes[text_idx], "Movie overviews, UMAP 3-D (colored by primary genre)")
fig.show()

## Takeaways

- The clusters you see depend on `n_neighbors` — but the *identity* of groups is remarkably consistent across the mid-range (5–30). That's your goldilocks zone.
- Very low `n_neighbors` (2–3) shatters manifolds into dozens of mini-clusters. Very high values (60+) collapse meaningful substructure.
- Posters cluster differently from overviews — not because one is wrong, but because *visual* and *narrative* similarity don't agree. This is the core insight of the talk.